# ImageNet-1K → private WebDataset parts on Kaggle

This CPU-only Kaggle notebook converts one deterministic part of the mounted ImageNet competition data into local WebDataset TAR shards, validates every shard, uploads the part as a **private** Kaggle Dataset, verifies the remote manifest, and only then enables cleanup. Use the same notebook repeatedly with `PART_ID=0..9`; do not make ten notebook copies.

## Goal and assumptions

- Attach the extracted `ILSVRC2012/<synset>/*.JPEG` Dataset shown in the Kaggle Input panel. The original CLS-LOC competition layout is also supported.
- Enable Internet only for cloning this repository and registering the private derived Dataset.
- Parts `0..9` divide the 1,000 training classes into ten contiguous, non-overlapping ranges. `PART_ID=10` is optional and only works for a CLS-LOC mount with labeled validation data.
- Each run stays below Kaggle's persistent-output allowance by holding only one part in `/kaggle/working`.
- Keep every derived Dataset private because ImageNet data is governed by its original access terms.

In [ ]:
from pathlib import Path
import csv, getpass, hashlib, json, os, shutil, subprocess, sys, tarfile, time

REPO_URL = 'https://github.com/Yang916-yy/LSSO.git'
BRANCH = 'experiment/vision-llama-bidirectional'
ROOT = Path('/kaggle/working/LSSO-code')
if not (ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle', 'kagglehub'], check=True)
if not os.environ.get('KAGGLE_API_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
        print('using KAGGLE_API_TOKEN from Kaggle Secrets')
    except Exception:
        os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Paste a Kaggle API token (input is hidden): ')
assert os.environ['KAGGLE_API_TOKEN'].strip(), 'Kaggle API token is empty'
print('repository commit:', subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 1. Select one part

Edit only `KAGGLE_USERNAME` and `PART_ID`. Run parts in any order. A completed local part is resumable, and a remotely verified part is skipped rather than uploaded as a replacement version.

In [ ]:
KAGGLE_USERNAME = 'Yang916-yy'  # exact Kaggle account name
PART_ID = 0                     # 0..9: train; 10 only for labeled CLS-LOC validation
NUM_TRAIN_PARTS = 10
SHARD_MAX_SAMPLES = 8192        # normally about 0.8–1.2 GiB per TAR
INPUT_ROOT = Path('/kaggle/input')  # bounded auto-discovery below

assert KAGGLE_USERNAME and KAGGLE_USERNAME != 'YOUR_KAGGLE_USERNAME'
assert 0 <= PART_ID <= NUM_TRAIN_PARTS
PART_NAME = 'validation' if PART_ID == NUM_TRAIN_PARTS else f'train-{PART_ID:02d}'
DATASET_SLUG = f'lsso-imagenet1k-wds-{PART_NAME}'
DATASET_HANDLE = f'{KAGGLE_USERNAME}/{DATASET_SLUG}'
PART_ROOT = Path('/kaggle/working') / DATASET_SLUG
VERIFY_ROOT = Path('/kaggle/temp') / f'verify-{DATASET_SLUG}'
print(json.dumps({
    'part_id': PART_ID, 'part_name': PART_NAME, 'handle': DATASET_HANDLE,
    'input': str(INPUT_ROOT), 'output': str(PART_ROOT),
}, indent=2))

## 2. Confirm the mounted ImageNet tree

This check is bounded: it tests the expected root and its immediate children rather than recursively walking two million files.

In [ ]:
assert INPUT_ROOT.is_dir(), f'Missing Kaggle Input root: {INPUT_ROOT}'
sys.path.insert(0, str(ROOT))
from tools.kaggle_imagenet_tree_to_wds import locate_imagenet, load_local_labels

CLS_LOC_ROOT, METADATA_ROOT = locate_imagenet(INPUT_ROOT)
SYNSETS, VALIDATION_LABELS = load_local_labels(METADATA_ROOT, CLS_LOC_ROOT)
FLAT_LAYOUT = not (CLS_LOC_ROOT / 'train').is_dir()
HAS_VALIDATION = bool(VALIDATION_LABELS) and (CLS_LOC_ROOT / 'val').is_dir()
assert not (PART_ID == NUM_TRAIN_PARTS and not HAS_VALIDATION), (
    'This flat ILSVRC2012 mount has no labeled validation split; use PART_ID=0..9.'
)
print('ImageNet root:', CLS_LOC_ROOT)
print('layout:', 'flat synset folders' if FLAT_LAYOUT else 'CLS-LOC')
print('classes:', len(SYNSETS), 'labeled validation:', HAS_VALIDATION)

## 3. Build or resume this part

Each TAR is written as `.partial` and atomically renamed only after closing. Completion markers make this cell resumable after interruption. Test images and XML annotations are never copied.

In [ ]:
build_started = time.time()
subprocess.run([
    sys.executable, '-u', str(ROOT / 'tools/kaggle_imagenet_tree_to_wds.py'),
    '--input', str(INPUT_ROOT), '--output', str(PART_ROOT),
    '--part-id', str(PART_ID), '--num-train-parts', str(NUM_TRAIN_PARTS),
    '--maxcount', str(SHARD_MAX_SAMPLES),
], check=True)
MANIFEST_PATH = PART_ROOT / 'manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text())
part_gib = sum(item['bytes'] for item in MANIFEST['shards']) / 2**30
assert part_gib < 19.0, f'Part is {part_gib:.2f} GiB; increase NUM_TRAIN_PARTS before upload'
print(f'part built/resumed in {(time.time() - build_started) / 60:.1f} min: {part_gib:.2f} GiB')

## 4. Validate local shards

This independently checks SHA-256, TAR readability, paired `.jpg`/`.cls` members, sample count, and the absence of partial files before any upload.

In [ ]:
assert not list(PART_ROOT.glob('*.partial'))
checked_samples = 0
for shard_record in MANIFEST['shards']:
    shard = PART_ROOT / shard_record['name']
    assert shard.stat().st_size == shard_record['bytes']
    digest = hashlib.sha256()
    with shard.open('rb', buffering=4 << 20) as stream:
        while block := stream.read(4 << 20):
            digest.update(block)
    assert digest.hexdigest() == shard_record['sha256']
    jpg_keys, cls_keys = set(), set()
    with tarfile.open(shard, 'r:') as archive:
        for member in archive:
            if member.isfile() and member.name.endswith('.jpg'):
                jpg_keys.add(member.name[:-4])
            elif member.isfile() and member.name.endswith('.cls'):
                cls_keys.add(member.name[:-4])
    assert jpg_keys == cls_keys and jpg_keys
    checked_samples += len(jpg_keys)
assert checked_samples == MANIFEST['samples']
LOCAL_VERIFIED = True
print(f'local verification passed: {len(MANIFEST["shards"])} shards, {checked_samples} samples')

## 5. Upload as a private Kaggle Dataset

The official CLI creates datasets as private unless `--public` is supplied; this notebook never supplies that flag. The setup cell reads `KAGGLE_API_TOKEN` from Kaggle Secrets or asks for it with hidden input. Existing handles are never silently versioned.

In [ ]:
assert globals().get('LOCAL_VERIFIED', False), 'Run local validation first'
metadata = {
    'title': f'LSSO ImageNet-1K WebDataset {PART_NAME}',
    'id': DATASET_HANDLE,
    'licenses': [{'name': 'other'}],
}
(PART_ROOT / 'dataset-metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')

def remote_files():
    result = subprocess.run(
        ['kaggle', 'datasets', 'files', DATASET_HANDLE, '--page-size', '200', '--csv'],
        text=True, capture_output=True,
    )
    if result.returncode != 0:
        return None
    return {row['name'] for row in csv.DictReader(result.stdout.splitlines())}

expected_remote = {item['name'] for item in MANIFEST['shards']} | {'manifest.json'}
listed = remote_files()
if listed is None:
    subprocess.run(['kaggle', 'datasets', 'create', '-p', str(PART_ROOT), '-q'], check=True)
elif not expected_remote.issubset(listed):
    raise RuntimeError(
        f'{DATASET_HANDLE} already exists but differs from this part; refusing to create a new version'
    )
else:
    print('matching remote Dataset already exists; upload skipped')

for attempt in range(80):
    listed = remote_files()
    if listed is not None and expected_remote.issubset(listed):
        break
    if attempt == 79:
        raise TimeoutError('Kaggle Dataset did not finish processing within 20 minutes')
    print(f'waiting for Kaggle processing: {attempt + 1}/80', flush=True)
    time.sleep(15)
print(f'remote file listing verified: {len(listed)} files')

## 6. Verify the remote manifest

A fresh copy of `manifest.json` is downloaded through KaggleHub and compared byte-for-byte. Cleanup remains disabled until this succeeds.

In [ ]:
import kagglehub
shutil.rmtree(VERIFY_ROOT, ignore_errors=True)
VERIFY_ROOT.mkdir(parents=True, exist_ok=True)
downloaded = Path(kagglehub.dataset_download(
    DATASET_HANDLE, path='manifest.json', output_dir=str(VERIFY_ROOT), force_download=True
))
remote_manifest = downloaded if downloaded.is_file() else downloaded / 'manifest.json'
if not remote_manifest.is_file():
    matches = list(VERIFY_ROOT.rglob('manifest.json'))
    assert len(matches) == 1, matches
    remote_manifest = matches[0]
assert remote_manifest.read_bytes() == MANIFEST_PATH.read_bytes()
REMOTE_VERIFIED = True
receipt = {
    'handle': DATASET_HANDLE, 'part_id': PART_ID, 'samples': MANIFEST['samples'],
    'bytes': sum(item['bytes'] for item in MANIFEST['shards']),
    'verified_at': time.strftime('%Y-%m-%dT%H:%M:%S%z'),
}
receipt_path = Path('/kaggle/working') / f'upload-receipt-{PART_NAME}.json'
receipt_path.write_text(json.dumps(receipt, indent=2) + '\n')
print('remote manifest verified:', DATASET_HANDLE)
print('receipt:', receipt_path)

## 7. Cleanup this local part

Run this only after the previous cell prints `remote manifest verified`. It removes the large local part but keeps the small upload receipt. It never touches `/kaggle/input`.

In [ ]:
assert globals().get('REMOTE_VERIFIED', False), 'Remote verification has not succeeded'
resolved_part = PART_ROOT.resolve()
working_root = Path('/kaggle/working').resolve()
assert resolved_part.parent == working_root and resolved_part.name.startswith('lsso-imagenet1k-wds-')
shutil.rmtree(resolved_part)
shutil.rmtree(VERIFY_ROOT, ignore_errors=True)
print('local part removed safely:', resolved_part)
print('next PART_ID:', PART_ID + 1 if PART_ID < NUM_TRAIN_PARTS else 'all parts complete')

## Next steps

Change `PART_ID` in the parameter cell and rerun sections 1–7. For the flat Dataset shown in the Input panel, the ten expected private handles are `train-00` through `train-09`; it does not provide labeled validation data. Keep the receipt JSON files until all parts are complete; they provide the handles, counts, and byte totals needed by the Colab downloader and final global-manifest check.